<a href="https://colab.research.google.com/github/dranphphmithe-ux/Book-Rental-System-Project/blob/main/%E0%B8%AA%E0%B8%B3%E0%B9%80%E0%B8%99%E0%B8%B2%E0%B8%82%E0%B8%AD%E0%B8%87_%E0%B8%AA%E0%B8%B3%E0%B9%80%E0%B8%99%E0%B8%B2%E0%B8%82%E0%B8%AD%E0%B8%87_book_rental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#ของน้ำหวาน
class Receipt:
    """คลาสคำนวณเงินและออกใบเสร็จ (คิดตามจริงไม่มีขั้นต่ำ, ใช้แต้มใหม่ลดราคาได้ทันที)"""

    def __init__(self,
        order_id: str,
        customer,
        items: list,
        days_rented: int,
        days_late: int = 0,
        use_points_promo: bool = True,):
        self.order_id = order_id
        self.customer = customer
        self.items = items
        self.days_rented = days_rented
        self.days_late = max(0, days_late)
        self.use_points_promo = use_points_promo
        self.date_issued = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def _calculate_rental_fee_by_days(self, days: int) -> float:
        """อัตราค่ายืม: 3 วัน 20 บาท, เศษวันละ 7 บาท (ไม่มีขั้นต่ำ)"""
        if days <= 0:
            return 0.0
        sets_of_3 = days // 3
        remaining_days = days % 3
        return (sets_of_3 * 20) + (remaining_days * 7)

    def calculate_totals(self) -> dict:
        total_books = len(self.items)

        # 1. คำนวณค่ายืมตามจริงก่อนหักส่วนลด (ไม่มีขั้นต่ำ 3 วันแล้ว)
        rental_fee_per_book = self._calculate_rental_fee_by_days(self.days_rented)
        total_rental_before_discount = total_books * rental_fee_per_book

        # 2. คำนวณแต้มที่ได้รับรอบนี้ (ยืม 1 เล่ม = 1 แต้ม, คืนตรงเวลา = +1)
        base_points = total_books
        on_time_bonus = 1 if self.days_late == 0 else 0
        earned_points = base_points + on_time_bonus

        # 3. คำนวณส่วนลด (รวมแต้มเดิม + แต้มใหม่ เพื่อแลกส่วนลด 10 แต้ม = ลด 7 บาท)
        total_discount = 0.0
        points_used = 0
        free_books_count = 0

        if self.use_points_promo:
            total_points_accumulated = self.customer.points + earned_points
            # ลดได้สูงสุดไม่เกินจำนวนเล่มที่ยืม
            free_books_count = min(total_points_accumulated // 10, total_books)
            total_discount = free_books_count * 7.0
            points_used = free_books_count * 10

        # 4. คำนวณยอดสุทธิและค่าปรับ
        total_rental_fee = max(0.0, total_rental_before_discount - total_discount)
        total_fine = total_books * self.days_late * 10
        grand_total = total_rental_fee + total_fine

        return {"total_books": total_books,
            "total_rental_before_discount": total_rental_before_discount,
            "free_books_count": free_books_count,
            "points_used": points_used,
            "total_rental_fee": total_rental_fee,
            "total_discount": total_discount,
            "total_fine": total_fine,
            "grand_total": grand_total,
            "earned_points": earned_points,}

    def print_receipt(self):
        calc = self.calculate_totals()

        # ตัดสต็อกหนังสือ
        for book in self.items:
            book.decrease_stock(1)

        # อัปเดตแต้มสมาชิก (บวกแต้มใหม่ก่อน แล้วหักแต้มที่ใช้เป็นส่วนลด)
        self.customer.add_points(calc["earned_points"])
        self.customer.points -= calc["points_used"]

        print("=" * 60)
        print(f"{'ใบเสร็จรับเงิน / Receipt':^60}")
        print("=" * 60)
        print(f"เลขที่ใบเสร็จ    : {self.order_id}")
        print(f"ชื่อลูกค้า       : คุณ{self.customer.name} (รหัส:"f" {self.customer.customer_id})")
        print(f"ระยะเวลาเช่า     : {self.days_rented} วัน")
        print("-" * 60)

        print(f"รายการหนังสือที่ยืม ({calc['total_books']} เล่ม):")
        for i, book in enumerate(self.items, 1):
            print(f"  [{i:02d}] {book.title} (ราคาปก {book.price:.0f} บาท)")

        print("-" * 60)
        print("รวมค่ายืมหนังสือ               :"f" {calc['total_rental_before_discount']:.2f} บาท")

        if calc["total_discount"] > 0:
            print(" ส่วนลดใช้แต้มแลกอ่านฟรี       :-"f"{calc['total_discount']:.2f} บาท")

        if calc["total_fine"] > 0:
            print(f" ค่าปรับคืนช้า ({self.days_late} วัน x {calc['total_books']} เล่ม"f" x 10B) : {calc['total_fine']:.2f} บาท")

        print("-" * 60)
        print(f"ยอดชำระสุทธิ (Grand Total)     : {calc['grand_total']:.2f} บาท")
        print("-" * 60)

        if calc["points_used"] > 0:
            print(f" ใช้แต้มแลกส่วนลด             : -{calc['points_used']} แต้ม")
        print(f" แต้มที่ได้รับครั้งนี้          : +{calc['earned_points']} แต้ม")
        print(f" แต้มสะสมคงเหลือปัจจุบัน      : {self.customer.points} แต้ม")
        print("=" * 60 + "\n")
